In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

df = pd.read_csv("accidents.csv")
# df.dtypes
# df.head(10)

features = [
    "Severity",
    "StartLat",
    "StartLng",
    "Distance",
    # "Zipcode",
]

X = df[features]
X.head()

,Severity,StartLat,StartLng,Distance
0,2,34.789009,-82.483383,0.000
1,2,40.998264,-76.650396,0.472
2,3,33.461292,-112.082001,0.000
3,2,42.288445,-87.924911,0.000
4,2,36.229259,-86.594650,0.000


In [2]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X_scaled.head()

,Severity,StartLat,StartLng,Distance
0,-0.613776,-0.352381,0.774024,-0.174263
1,-0.613776,0.915880,1.110081,0.111248
2,1.200680,-0.623572,-0.931244,-0.174263
3,-0.613776,1.179404,0.460521,-0.174263
4,-0.613776,-0.058205,0.537161,-0.174263


In [3]:
inertia = []

for k in range(2, 11):
    model = KMeans(
        n_clusters=k,
        random_state=42
    )
    model.fit(X_scaled)
    inertia.append(model.inertia_)

elbow_df = pd.DataFrame({
    "k": range(2, 11),
    "Inertia": inertia
})

fig = px.line(
    elbow_df,
    x="k",
    y="Inertia",
    markers=True,
    title="Elbow Method for Choosing the Number of Clusters"
)

fig.update_layout(
    xaxis_title="Number of Clusters (k)",
    yaxis_title="Inertia",
    template="plotly_white"
)

fig.show()

In [5]:
scores = []

for k in range(2, 11):
    model = KMeans(
        n_clusters=k,
        random_state=42
    )

    labels = model.fit_predict(X_scaled)

    scores.append(
        silhouette_score(X_scaled, labels)
    )

silhouette_df = pd.DataFrame({
    "k": range(2, 11),
    "Silhouette Score": scores
})

fig = px.line(
    silhouette_df,
    x="k",
    y="Silhouette Score",
    markers=True,
    title="Silhouette Scores for Different Numbers of Clusters"
)

fig.update_layout(
    xaxis_title="Number of Clusters (k)",
    yaxis_title="Silhouette Score",
    template="plotly_white"
)

fig.show()

In [6]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

k_range = range(2, 11)
inertia = []
silhouette_scores = []

for k in k_range:
    model = KMeans(
        n_clusters=k,
        random_state=42
    )
    labels = model.fit_predict(X_scaled)
    inertia.append(model.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Elbow Method", "Silhouette Score")
)

fig.add_trace(
    go.Scatter(
        x=list(k_range), y=inertia,
        mode="lines+markers",
        name="Inertia",
        line=dict(color="#2a78d6", width=2),
        marker=dict(size=8)
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=list(k_range), y=silhouette_scores,
        mode="lines+markers",
        name="Silhouette Score",
        line=dict(color="#1baf7a", width=2),
        marker=dict(size=8)
    ),
    row=1, col=2
)

fig.update_xaxes(title_text="Number of Clusters (k)", row=1, col=1)
fig.update_xaxes(title_text="Number of Clusters (k)", row=1, col=2)
fig.update_yaxes(title_text="Inertia", row=1, col=1)
fig.update_yaxes(title_text="Silhouette Score", row=1, col=2)

fig.update_layout(
    template="plotly_white",
    title_text="Choosing the Number of Clusters (k)",
    showlegend=False,
    height=450,
    width=1000
)

fig.show()

In [27]:
kmeans = KMeans(
    n_clusters=2,
    random_state=42
)

df["Cluster"] = kmeans.fit_predict(X_scaled)
df["Cluster"] = df["Cluster"].astype("category")

In [28]:
pca = PCA(n_components=2)

components = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    components,
    columns=["PC1", "PC2"]
)

pca_df["Cluster"] = df["Cluster"]

fig = px.scatter(
    pca_df,
    x="PC1",
    y="PC2",
    color="Cluster",
    title="K-Means Clusters (PCA Projection)",
    opacity=0.8,
    height=700
)

fig.update_layout(
    template="plotly_white"
)

fig.show()

In [29]:
cluster_summary = (
    df
    .groupby("Cluster")[features]
    .mean()
    .round(3)
)

cluster_summary

/var/folders/89/f_fc311s7v977_m_np2qc56c0000gn/T/ipykernel_6853/3313565994.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("Cluster")[features]


,Severity,StartLat,StartLng,Distance
Cluster,,,,
0,2.282,37.148,-118.252,0.183
1,2.367,36.190,-84.487,0.342


In [30]:
fig = px.box(
    df,
    x="Cluster",
    y="Severity",
    color="Cluster",
    title="Severity by Cluster"
)

fig.update_layout(
    template="plotly_white",
    showlegend=False
)

fig.show()

In [34]:
cluster_counts = (
    df["Cluster"]
    .value_counts()
    .sort_index()
    .reset_index()
)

cluster_counts.columns = ["Cluster", "Severity"]

fig = px.bar(
    cluster_counts,
    x="Cluster",
    y="Severity",
    color="Cluster",
    title="Number of Accidents by Cluster",
    text="Severity"
)

fig.update_layout(
    template="plotly_white",
    showlegend=False
)

fig.show()